In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, classification_report, roc_curve, confusion_matrix,accuracy_score
from sklearn.preprocessing import MinMaxScaler,LabelEncoder
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import seaborn as sn

In [2]:
df=pd.read_csv('../../dataset/diabetes_dataset.csv')
df.head()

,year,gender,age,location,race:AfricanAmerican,race:Asian,race:Caucasian,race:Hispanic,race:Other,hypertension,heart_disease,smoking_history,bmi,hbA1c_level,blood_glucose_level,diabetes
0,2020,Female,32.0,Alabama,0,0,0,0,1,0,0,never,27.32,5.0,100,0
1,2015,Female,29.0,Alabama,0,1,0,0,0,0,0,never,19.95,5.0,90,0
2,2015,Male,18.0,Alabama,0,0,0,0,1,0,0,never,23.76,4.8,160,0
3,2015,Male,41.0,Alabama,0,0,1,0,0,0,0,never,27.32,4.0,159,0
4,2016,Female,52.0,Alabama,1,0,0,0,0,0,0,never,23.75,6.5,90,0


In [3]:
df.drop(columns=['year','location','race:AfricanAmerican','race:Asian','race:Caucasian','race:Hispanic','race:Other','smoking_history'],inplace=True)
df

,gender,age,hypertension,heart_disease,bmi,hbA1c_level,blood_glucose_level,diabetes
0,Female,32.0,0,0,27.32,5.0,100,0
1,Female,29.0,0,0,19.95,5.0,90,0
2,Male,18.0,0,0,23.76,4.8,160,0
3,Male,41.0,0,0,27.32,4.0,159,0
4,Female,52.0,0,0,23.75,6.5,90,0
...,...,...,...,...,...,...,...,...
99995,Female,33.0,0,0,21.21,6.5,90,0
99996,Female,80.0,0,0,36.66,5.7,100,0
99997,Male,46.0,0,0,36.12,6.2,158,0
99998,Female,51.0,0,0,29.29,6.0,155,0


In [4]:
encoder = LabelEncoder()
df['gender'] = encoder.fit_transform(df['gender'])

label_mapping = dict(zip(encoder.classes_, range(len(encoder.classes_))))

print("Label Encoding Mapping:", label_mapping)


Label Encoding Mapping: {'Female': 0, 'Male': 1, 'Other': 2}


In [5]:
df.drop_duplicates(inplace=True)
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 91313 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   gender               91313 non-null  int32  
 1   age                  91313 non-null  float64
 2   hypertension         91313 non-null  int64  
 3   heart_disease        91313 non-null  int64  
 4   bmi                  91313 non-null  float64
 5   hbA1c_level          91313 non-null  float64
 6   blood_glucose_level  91313 non-null  int64  
 7   diabetes             91313 non-null  int64  
dtypes: float64(3), int32(1), int64(4)
memory usage: 5.9 MB
None


In [6]:
## removing of missing values 
df = df.drop(df[df['gender'] == 2].index)
df.shape

(91295, 8)

In [7]:
# Select columns to normalize
cols_to_normalize = ['age', 'bmi', 'hbA1c_level', 'blood_glucose_level']

# Initialize scaler and apply normalization
scaler = MinMaxScaler()
df[cols_to_normalize] = scaler.fit_transform(df[cols_to_normalize])

print(df.tail())

       gender       age  hypertension  heart_disease       bmi  hbA1c_level  \
99995       0  0.411912             0              0  0.130719     0.545455   
99996       0  1.000000             0              0  0.311041     0.400000   
99997       1  0.574575             0              0  0.304739     0.490909   
99998       0  0.637137             0              0  0.225023     0.454545   
99999       1  0.161662             0              0  0.083450     0.272727   

       blood_glucose_level  diabetes  
99995             0.045455         0  
99996             0.090909         0  
99997             0.354545         0  
99998             0.340909         0  
99999             0.045455         0  


In [8]:
## here we will perform the interaction
# Binary feature columns
from itertools import product
df['age_high'] = (df['age'] > 60).astype(int)
df['bmi_obese'] = (df['bmi'] > 23).astype(int)
df['hba1c_diabetes'] = (df['hbA1c_level'] > 6.5).astype(int)
df['glucose_high'] = (df['blood_glucose_level'] > 126).astype(int)

binary_cols = ['age_high', 'heart_disease', 'hba1c_diabetes', 'hypertension', 'bmi_obese', 'gender', 'glucose_high']

# Interaction for hbA1c_level with all others except hbA1c_level and blood_glucose_level
for col in binary_cols:
    if col not in ['hba1c_diabetes', 'glucose_high']:
        for val1, val2 in product([0, 1], repeat=2):
            new_col = f"hbA1c_level({val1}){col}({val2})"
            df[new_col] = ((df['hba1c_diabetes'] == val1) & (df[col] == val2)).astype(int)

df['hbA1c_level(0)_glucose_high(0)'] = ((df['glucose_high'] == 0) & (df['hba1c_diabetes'] == 0)).astype(int)
df['hbA1c_level(0)_glucose_high(1)'] = ((df['glucose_high'] == 0) & (df['hba1c_diabetes'] == 1)).astype(int)
df['hbA1c_level(1)_glucose_high(0)'] = ((df['glucose_high'] == 1) & (df['hba1c_diabetes'] == 0)).astype(int)
df['hbA1c_level(1)_glucose_high(1)'] = ((df['glucose_high'] == 1) & (df['hba1c_diabetes'] == 1)).astype(int)
# Interaction for blood_glucose_level with all others except hbA1c_level and blood_glucose_level
for col in binary_cols:
    if col not in ['hba1c_diabetes', 'glucose_high']:
        for val1, val2 in product([0, 1], repeat=2):
            new_col = f"glucose_high({val1}){col}({val2})"
            df[new_col] = ((df['glucose_high'] == val1) & (df[col] == val2)).astype(int)
            


df.drop(columns=['age_high','bmi_obese','hba1c_diabetes','glucose_high'],inplace=True)
df

,gender,age,hypertension,heart_disease,bmi,hbA1c_level,blood_glucose_level,diabetes,hbA1c_level(0)age_high(0),hbA1c_level(0)age_high(1),...,glucose_high(1)hypertension(0),glucose_high(1)hypertension(1),glucose_high(0)bmi_obese(0),glucose_high(0)bmi_obese(1),glucose_high(1)bmi_obese(0),glucose_high(1)bmi_obese(1),glucose_high(0)gender(0),glucose_high(0)gender(1),glucose_high(1)gender(0),glucose_high(1)gender(1)
0,0,0.399399,0,0,0.202031,0.272727,0.090909,0,1,0,...,0,0,1,0,0,0,1,0,0,0
1,0,0.361862,0,0,0.116013,0.272727,0.045455,0,1,0,...,0,0,1,0,0,0,1,0,0,0
2,1,0.224224,0,0,0.160481,0.236364,0.363636,0,1,0,...,0,0,1,0,0,0,0,1,0,0
3,1,0.512012,0,0,0.202031,0.090909,0.359091,0,1,0,...,0,0,1,0,0,0,0,1,0,0
4,0,0.649650,0,0,0.160364,0.545455,0.045455,0,1,0,...,0,0,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,0,0.411912,0,0,0.130719,0.545455,0.045455,0,1,0,...,0,0,1,0,0,0,1,0,0,0
99996,0,1.000000,0,0,0.311041,0.400000,0.090909,0,1,0,...,0,0,1,0,0,0,1,0,0,0
99997,1,0.574575,0,0,0.304739,0.490909,0.354545,0,1,0,...,0,0,1,0,0,0,0,1,0,0
99998,0,0.637137,0,0,0.225023,0.454545,0.340909,0,1,0,...,0,0,1,0,0,0,1,0,0,0


In [9]:
from sklearn.feature_selection import mutual_info_classif
# Filter only interaction columns
interaction_cols = [col for col in df.columns if col.startswith('glucose_high(') or col.startswith('hbA1c_level(')]
y = df['diabetes']
# Compute mutual information
mi_scores = mutual_info_classif(df[interaction_cols], y, discrete_features=True)
# len(mi_scores)
mi_series = pd.Series(mi_scores, index=interaction_cols).sort_values(ascending=False)
top_features = mi_series.head(10).index

print(top_features)

Index(['hbA1c_level(0)hypertension(0)', 'hbA1c_level(0)hypertension(1)',
       'glucose_high(0)hypertension(0)', 'glucose_high(0)hypertension(1)',
       'hbA1c_level(0)heart_disease(0)', 'hbA1c_level(0)heart_disease(1)',
       'glucose_high(0)heart_disease(0)', 'glucose_high(0)heart_disease(1)',
       'hbA1c_level(0)gender(0)', 'glucose_high(0)gender(1)'],
      dtype='object')


In [10]:
df.drop(columns=[col for col in df.columns if col in interaction_cols and col not in top_features],inplace=True)
df

,gender,age,hypertension,heart_disease,bmi,hbA1c_level,blood_glucose_level,diabetes,hbA1c_level(0)heart_disease(0),hbA1c_level(0)heart_disease(1),hbA1c_level(0)hypertension(0),hbA1c_level(0)hypertension(1),hbA1c_level(0)gender(0),glucose_high(0)heart_disease(0),glucose_high(0)heart_disease(1),glucose_high(0)hypertension(0),glucose_high(0)hypertension(1),glucose_high(0)gender(1)
0,0,0.399399,0,0,0.202031,0.272727,0.090909,0,1,0,1,0,1,1,0,1,0,0
1,0,0.361862,0,0,0.116013,0.272727,0.045455,0,1,0,1,0,1,1,0,1,0,0
2,1,0.224224,0,0,0.160481,0.236364,0.363636,0,1,0,1,0,0,1,0,1,0,1
3,1,0.512012,0,0,0.202031,0.090909,0.359091,0,1,0,1,0,0,1,0,1,0,1
4,0,0.649650,0,0,0.160364,0.545455,0.045455,0,1,0,1,0,1,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,0,0.411912,0,0,0.130719,0.545455,0.045455,0,1,0,1,0,1,1,0,1,0,0
99996,0,1.000000,0,0,0.311041,0.400000,0.090909,0,1,0,1,0,1,1,0,1,0,0
99997,1,0.574575,0,0,0.304739,0.490909,0.354545,0,1,0,1,0,0,1,0,1,0,1
99998,0,0.637137,0,0,0.225023,0.454545,0.340909,0,1,0,1,0,1,1,0,1,0,0


In [11]:
from imblearn.over_sampling import SMOTE
x=df.drop(columns=['diabetes'])
y=df['diabetes']
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [17]:
# Create a polynomial kernel SVM with degree 5 (default)
poly_svm = SVC(C=14, degree=5, gamma=1, kernel='poly')

# Train the model
poly_svm.fit(X_train, y_train)

SVC(C=14, degree=5, gamma=1, kernel='poly')

In [18]:
 ## accuracy, precision, recall, and the F1-score.
y_pred = poly_svm.predict(X_test)

In [19]:
cm = confusion_matrix(y_test, y_pred) 
cm

array([[14660,  1852],
       [  155,  1592]], dtype=int64)

In [20]:
accuracy=accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")

Accuracy: 0.8900816035927488
Precision: 0.462253193960511
Recall: 0.911276473955352
F1-score: 0.6133692930071277
